In [24]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
YOUTUBE_VIDEO = "https://www.youtube.com/watch?v=eE7CamOE-PA"

In [2]:
from langchain_openai.chat_models import ChatOpenAI

model = ChatOpenAI(api_key=OPENAI_API_KEY, model="gpt-3.5-turbo")

In [3]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [ ]:
from langchain.prompts import ChatPromptTemplate

template = """
Answer the question based on the context below. If you can't 
answer the question, reply "I don't know".

Context: {context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [33]:
import re
pattern = r"(?:v=|youtu\.be/)([a-zA-Z0-9_-]{11})"
match = re.search(pattern, YOUTUBE_VIDEO)
print(match.group(1))    


eE7CamOE-PA


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
ytt_api = YouTubeTranscriptApi()

transcript = ytt_api.fetch(match.group(1))

# extract only text
text = " ".join([snippet.text for snippet in transcript.snippets])

# save to file
with open("transcription.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("transcription.txt")
text_documents = loader.load()

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)
documents = text_splitter.split_documents(text_documents)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
embeddings = OpenAIEmbeddings()
vectorstore2 = DocArrayInMemorySearch.from_documents(documents, embeddings)

In [16]:
from langchain_pinecone import PineconeVectorStore

index_name = "youtube-index"

pinecone = PineconeVectorStore.from_documents(
    documents, embeddings, index_name=index_name
)

c:\Users\mmuba\anaconda3\envs\RAG\lib\site-packages\pinecone\data\index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [23]:
chain = (
    {"context": pinecone.as_retriever(), "question": RunnablePassthrough()}
    | prompt
    | model
    | parser
)

chain.invoke("Did Andrej Karpathy work in tesla? And in which part is it discussed?")

'Yes, Andrej Karpathy did work at Tesla. This is discussed in the context where it mentions "aside from a potential Godfather part two, with the AGI at Tesla and beyond, what does the future Fondja Karpathy".'